In [155]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [156]:
wc2026_teams = {
    # UEFA (16)
    'England'               : 'UEFA',
    'France'                : 'UEFA',
    'Germany'               : 'UEFA',
    'Spain'                 : 'UEFA',
    'Portugal'              : 'UEFA',
    'Netherlands'           : 'UEFA',
    'Belgium'               : 'UEFA',
    'Croatia'               : 'UEFA',
    'Switzerland'           : 'UEFA',
    'Norway'                : 'UEFA',
    'Scotland'              : 'UEFA',
    'Austria'               : 'UEFA',
    'Czech Republic'        : 'UEFA',
    'Bosnia and Herzegovina': 'UEFA',
    'Sweden'                : 'UEFA',
    'Turkey'                : 'UEFA',
 
    # CAF (10)
    'Morocco'               : 'CAF',
    'Egypt'                 : 'CAF',
    'Algeria'               : 'CAF',
    'Ghana'                 : 'CAF',
    'Ivory Coast'           : 'CAF',
    'Tunisia'               : 'CAF',
    'Senegal'               : 'CAF',
    'South Africa'          : 'CAF',
    'DR Congo'              : 'CAF',
    'Cape Verde'            : 'CAF',   # World Cup debut
 
    # AFC (9)
    'Iran'                  : 'AFC',
    'Iraq'                  : 'AFC',   # via inter-confederation playoff
    'Saudi Arabia'          : 'AFC',
    'Jordan'                : 'AFC',   # World Cup debut
    'Qatar'                 : 'AFC',
    'Uzbekistan'            : 'AFC',   # World Cup debut
    'Japan'                 : 'AFC',
    'South Korea'           : 'AFC',
    'Australia'             : 'AFC',
 
    # CONMEBOL (6)
    'Argentina'             : 'CONMEBOL',
    'Brazil'                : 'CONMEBOL',
    'Colombia'              : 'CONMEBOL',
    'Ecuador'               : 'CONMEBOL',
    'Paraguay'              : 'CONMEBOL',
    'Uruguay'               : 'CONMEBOL',
 
    # CONCACAF (6)
    'United States'         : 'CONCACAF',
    'Mexico'                : 'CONCACAF',
    'Canada'                : 'CONCACAF',
    'Panama'                : 'CONCACAF',
    'Haiti'                 : 'CONCACAF',
    'Curacao'               : 'CONCACAF',  # World Cup debut (smallest nation ever)
 
    # OFC (1)
    'New Zealand'           : 'OFC',
}

In [157]:
print(f"Total 2026 qualified nations: {len(wc2026_teams)}")

Total 2026 qualified nations: 48


In [158]:
name_fixes = {
    'Bosnia-Herzegovina'    : 'Bosnia and Herzegovina',
    'Ivory Coast'           : 'Ivory Coast',   # may appear as Côte d\'Ivoire
    "Côte d'Ivoire"         : 'Ivory Coast',
    'DR Congo'              : 'DR Congo',       # may appear as Congo DR
    'Congo DR'              : 'DR Congo',
    'Turkey'                : 'Turkey',         # may appear as Türkiye
    'Türkiye'               : 'Turkey',
    'Korea Republic'        : 'South Korea',
    'USA'                   : 'United States',
    'Curaçao'               : 'Curacao',
    'Cape Verde Islands'    : 'Cape Verde',
}

In [159]:
world_cup_2026 = pd.read_csv("C:/Users/laila/Documents/Football Project/results.csv")

In [160]:
world_cup_2026.columns = world_cup_2026.columns.str.replace('_','').str.upper()
world_cup_2026['DATE'] = pd.to_datetime(world_cup_2026['DATE'], errors='coerce')
world_cup_2026['HOMETEAM'] = world_cup_2026['HOMETEAM'].replace(name_fixes)
world_cup_2026['AWAYTEAM'] = world_cup_2026['AWAYTEAM'].replace(name_fixes)
world_cup_2026 = world_cup_2026[world_cup_2026['DATE'] >= '2006-01-01']
world_cup_2026 = world_cup_2026[world_cup_2026['TOURNAMENT'] != 'Friendly']
 
print(f"Rows after filter: {len(world_cup_2026)}")

Rows after filter: 12945


In [161]:
home1 = world_cup_2026[['HOMETEAM', 'AWAYTEAM', 'HOMESCORE', 'AWAYSCORE']].copy()
home1.columns = ['TEAM', 'OPPONENT', 'GOALS_SCORED', 'GOALS_CONCEDED']
away1 = world_cup_2026[['AWAYTEAM','HOMETEAM','AWAYSCORE','HOMESCORE']].copy()
away1.columns = ['TEAM','OPPONENT','GOALS_SCORED','GOALS_CONCEDED']

In [162]:
matches = pd.concat([home1, away1], ignore_index=True)
matches['WIN']         = (matches['GOALS_SCORED'] > matches['GOALS_CONCEDED']).astype(int)
matches['DRAW']        = (matches['GOALS_SCORED'] == matches['GOALS_CONCEDED']).astype(int)
matches['LOSS']        = (matches['GOALS_SCORED'] < matches['GOALS_CONCEDED']).astype(int)
matches['CLEAN_SHEET'] = (matches['GOALS_CONCEDED'] == 0).astype(int)

In [163]:
summary = matches.groupby('TEAM').agg(
    matches_played = ('TEAM',          'count'),
    goals_scored    = ('GOALS_SCORED',   'sum'),
    goals_conceded  = ('GOALS_CONCEDED', 'sum'),
    clean_sheets    = ('CLEAN_SHEET',    'sum'),
    wins            = ('WIN',            'sum'),
    draws           = ('DRAW',           'sum'),
    losses          = ('LOSS',           'sum'),
).reset_index()

In [164]:
summary['goals_per_game']          = summary['goals_scored']   / summary['matches_played']
summary['goals_conceded_per_game'] = summary['goals_conceded'] / summary['matches_played']
summary['goal_diff_per_game']      = (summary['goals_scored'] - summary['goals_conceded']) / summary['matches_played']
summary['cleansheet_pct']          = summary['clean_sheets']   / summary['matches_played']
summary['win_rate']                = summary['wins']           / summary['matches_played']

In [165]:
wc_only = world_cup_2026[world_cup_2026['TOURNAMENT'] == 'FIFA World Cup'].copy()
wc_only['YEAR'] = wc_only['DATE'].dt.year


In [166]:
wc_home = wc_only[['YEAR', 'HOMETEAM']].rename(columns={'HOMETEAM': 'TEAM'})
wc_away = wc_only[['YEAR', 'AWAYTEAM']].rename(columns={'AWAYTEAM':'TEAM'})
wc_teams_df = pd.concat([wc_home, wc_away], ignore_index=True)
wc_appearances = (
    wc_teams_df.drop_duplicates(subset=['TEAM','YEAR'])
               .groupby('TEAM')['YEAR']
               .nunique()
               .reset_index()
               .rename(columns={'YEAR':'wc_appearances'})
)
 
summary = summary.merge(wc_appearances, on='TEAM', how='left')
summary['wc_appearances'] = summary['wc_appearances'].fillna(0).astype(int)

In [167]:
qualified_nations = list(wc2026_teams.keys())
summary_48 = summary[summary['TEAM'].isin(qualified_nations)].copy()
summary_48['confederation'] = summary_48['TEAM'].map(wc2026_teams)

In [168]:
matched = summary_48['TEAM'].tolist()
missing = [t for t in qualified_nations if t not in matched]
print(f"\nQualified nations missing from results.csv: {missing}")
print(f"Nations with match data: {len(summary_48)} / {len(qualified_nations)}")


Qualified nations missing from results.csv: []
Nations with match data: 48 / 48


In [169]:
for nation in missing:
    conf = wc2026_teams[nation]
    conf_avg = summary_48[summary_48['confederation'] == conf].mean(numeric_only=True)
    new_row = conf_avg.to_dict()
    new_row['TEAM'] = nation
    new_row['confederation'] = conf
    new_row['matches_played'] = 0 
    new_row['wc_appearances'] = 0
    summary_48 = pd.concat([summary_48, pd.DataFrame([new_row])], ignore_index=True)

In [170]:
recent = world_cup_2026[world_cup_2026['DATE'] >= '2022-01-01'].copy()
home_re = recent[['HOMETEAM','AWAYTEAM','HOMESCORE','AWAYSCORE']].copy()
home_re.columns = ['TEAM','OPPONENT','GOALS_SCORED','GOALS_CONCEDED']
away_re = recent[['AWAYTEAM','HOMETEAM','AWAYSCORE','HOMESCORE']].copy()
away_re.columns = ['TEAM','OPPONENT','GOALS_SCORED','GOALS_CONCEDED']
matches_re = pd.concat([home_re, away_re], ignore_index=True)
matches_re['WIN'] = (matches_re['GOALS_SCORED'] > matches_re['GOALS_CONCEDED']).astype(int)

In [171]:
recent_form = matches_re.groupby('TEAM').agg(
    recent_matches = ('TEAM', 'count'),
    recent_wins    = ('WIN', 'sum'),
).reset_index()
recent_form['recent_win_rate'] = recent_form['recent_wins'] / recent_form['recent_matches']

In [172]:
summary_48 = summary_48.merge(recent_form[['TEAM','recent_win_rate']], on='TEAM', how='left')
summary_48['recent_win_rate'] = summary_48['recent_win_rate'].fillna(0)

In [174]:
wc2022_stage = {
    # Winner & finalists
    'Argentina'    : 7,  # Winner
    'France'       : 6,  # Runner-up
    'Croatia'      : 5,  # 3rd place
    'Morocco'      : 4,  # 4th place

    # Quarter-finals
    'Netherlands'  : 3,
    'Brazil'       : 3,
    'England'      : 3,
    'Portugal'     : 3,

    # Round of 16
    'Japan'        : 2,
    'South Korea'  : 2,
    'Australia'    : 2,
    'Senegal'      : 2,
    'Poland'       : 2,
    'Switzerland'  : 2,
    'United States': 2,
    'Spain'        : 2,

    # Group stage 
    'Germany'      : 1,
    'Belgium'      : 1,
    'Uruguay'      : 1,
    'Mexico'       : 1,
    'Denmark'      : 1,  # not in 2026
    'Tunisia'      : 1,
    'Ecuador'      : 1,
    'Cameroon'     : 1,
    'Serbia'       : 1,  # not in 2026
    'Ghana'        : 1,
    'Iran'         : 1,
    'Saudi Arabia' : 1,
    'Canada'       : 1,
    'Costa Rica'   : 1,  # not in 2026
    'Wales'        : 1,  # not in 2026
    'Qatar'        : 1,
}
 
summary_48['wc2022_stage'] = summary_48['TEAM'].map(wc2022_stage).fillna(0)

In [175]:
conf_strength = {
    'UEFA'    : 1.00,
    'CONMEBOL': 0.98,
    'CAF'     : 0.88,
    'AFC'     : 0.80,
    'CONCACAF': 0.82,
    'OFC'     : 0.75,
}

summary_48['conf_multiplier'] = summary_48['confederation'].map(conf_strength)
summary_48['goal_diff_per_game'] = summary_48['goal_diff_per_game'] * summary_48['conf_multiplier']
summary_48['win_rate']           = summary_48['win_rate']           * summary_48['conf_multiplier']
summary_48['goals_per_game']     = summary_48['goals_per_game']     * summary_48['conf_multiplier']
summary_48['cleansheet_pct']     = summary_48['cleansheet_pct']     * summary_48['conf_multiplier']


In [176]:
indicators = ['goals_per_game', 'goal_diff_per_game', 'cleansheet_pct',
              'win_rate', 'wc_appearances', 'recent_win_rate', 'wc2022_stage']
scaler = MinMaxScaler()
scaled = scaler.fit_transform(summary_48[indicators].fillna(0))
scaled_df = pd.DataFrame(scaled, columns=[m + '_scaled' for m in indicators])
weights = {
    'goals_per_game_scaled'    : 0.10,
    'goal_diff_per_game_scaled': 0.10,
    'cleansheet_pct_scaled'    : 0.15,
    'win_rate_scaled'          : 0.20,
    'wc_appearances_scaled'    : 0.10,
    'recent_win_rate_scaled'   : 0.15,
    'wc2022_stage_scaled'      : 0.20,
}
 
print("Weights sum:", sum(weights.values()))

Weights sum: 1.0


In [177]:
summary_48 = summary_48.reset_index(drop=True)
summary_48['composite_score'] = sum(scaled_df[col] * w for col, w in weights.items())
summary_48 = summary_48.sort_values('composite_score', ascending=False).reset_index(drop=True)
summary_48['rank'] = summary_48.index + 1

In [178]:
print("\n=== ALL 48 QUALIFIED NATIONS — RANKED ===")
print(summary_48[['rank','TEAM','confederation','matches_played',
                   'goal_diff_per_game','win_rate','recent_win_rate',
                   'wc2022_stage','composite_score']].to_string(index=False))


=== ALL 48 QUALIFIED NATIONS — RANKED ===
 rank                   TEAM confederation  matches_played  goal_diff_per_game  win_rate  recent_win_rate  wc2022_stage  composite_score
    1                  Spain          UEFA             181            1.646409  0.729282         0.697674           2.0         0.818764
    2                England          UEFA             164            1.640244  0.640244         0.625000           3.0         0.795041
    3              Argentina      CONMEBOL             165            0.831515  0.564242         0.694444           7.0         0.769070
    4                 France          UEFA             167            1.065868  0.610778         0.604651           6.0         0.755581
    5               Portugal          UEFA             181            1.237569  0.613260         0.681818           3.0         0.724844
    6            Netherlands          UEFA             163            1.368098  0.656442         0.581395           3.0         0.69028

In [179]:
summary_48.to_csv(
    "C:/Users/laila/Documents/Football Project/wc2026_all48_summary.csv",
    index=False
)
print("\nSaved: wc2026_all48_summary.csv")


Saved: wc2026_all48_summary.csv


In [1]:
import pandas as pd

flags = {
    'Spain': '🇪🇸', 'England': '🏴󠁧󠁢󠁥󠁮󠁧󠁿', 'France': '🇫🇷',
    'Argentina': '🇦🇷', 'Brazil': '🇧🇷', 'Portugal': '🇵🇹',
    'Netherlands': '🇳🇱', 'Germany': '🇩🇪', 'Morocco': '🇲🇦',
    'Croatia': '🇭🇷', 'Japan': '🇯🇵', 'Belgium': '🇧🇪',
    'Iran': '🇮🇷', 'Australia': '🇦🇺', 'Senegal': '🇸🇳',
    'South Korea': '🇰🇷', 'Mexico': '🇲🇽', 'Ivory Coast': '🇨🇮',
    'United States': '🇺🇸', 'Switzerland': '🇨🇭', 'Algeria': '🇩🇿',
    'Egypt': '🇪🇬', 'Tunisia': '🇹🇳', 'Norway': '🇳🇴',
    'Uruguay': '🇺🇾', 'New Zealand': '🇳🇿', 'Uzbekistan': '🇺🇿',
    'Turkey': '🇹🇷', 'Ghana': '🇬🇭', 'Canada': '🇨🇦',
    'Sweden': '🇸🇪', 'Colombia': '🇨🇴', 'Austria': '🇦🇹',
    'Czech Republic': '🇨🇿', 'Jordan': '🇯🇴', 'Saudi Arabia': '🇸🇦',
    'Scotland': '🏴󠁧󠁢󠁳󠁣󠁴󠁿', 'Haiti': '🇭🇹', 'DR Congo': '🇨🇩',
    'South Africa': '🇿🇦', 'Qatar': '🇶🇦', 'Panama': '🇵🇦',
    'Cape Verde': '🇨🇻', 'Ecuador': '🇪🇨',
    'Bosnia and Herzegovina': '🇧🇦', 'Iraq': '🇮🇶',
    'Curacao': '🇨🇼', 'Paraguay': '🇵🇾',
}

# Check actual column names in each file first
files = [
    'wc2026_all48_summary',
    'monte_carlo_results_2026',
    'top20_contenders',
    'group_predictions',
    'stage_progression',
]

for filename in files:
    path = f"C:/Users/laila/Documents/Football Project/{filename}.csv"
    df = pd.read_csv(path)
    
    # Auto detect team column
    team_col = None
    for col in df.columns:
        if col.upper() == 'TEAM':
            team_col = col
            break
    
    if team_col is None:
        print(f"✗ {filename} — no TEAM column found. Columns: {df.columns.tolist()}")
        continue
    
    df['Flag'] = df[team_col].map(flags).fillna('')
    df['Team with Flag'] = df['Flag'] + ' ' + df[team_col]
    df.to_csv(path, index=False)
    print(f"✓ {filename} — used column '{team_col}'")

print("\nAll done!")

✓ wc2026_all48_summary — used column 'TEAM'
✓ monte_carlo_results_2026 — used column 'TEAM'
✓ top20_contenders — used column 'TEAM'
✓ group_predictions — used column 'Team'
✓ stage_progression — used column 'TEAM'

All done!
